CAP5619-26Fall 0061
Marcelo G. Proaño MA423904
Assignment 2 - LLM Document Analysis

Setup & Ticker Retrieval
Obtain CIK numbers for companies: This block of code imports necessary libraries and fetches the top companies giving us over 100 S&P 500 level companies to process by parsing the JSON and formating the CIK numbers (with leading zeros needed for later SEC links).

In [5]:
import requests
import json
import pandas as pd
import re
import time
from bs4 import BeautifulSoup

# Setup headers for SEC EDGAR
headers = {
    'User-Agent': 'Marcelo G. Proano ma423904@ucf.edu' 
}

# Fetch the company tickers and CIKs
tickers_url = "https://www.sec.gov/files/company_tickers.json"
response = requests.get(tickers_url, headers=headers)

if response.status_code == 200:
    tickers_data = response.json()
    
    # The SEC returns a dictionary of dictionaries, convert it to a list.
    companies = []
    for key, value in tickers_data.items():
        # Pad the CIK with leading zeros to make it 10 digits (required by some SEC endpoints)
        cik_str = str(value['cik_str']).zfill(10)
        companies.append({
            'cik': cik_str,
            'ticker': value['ticker'],
            'title': value['title']
        })
    
    # Put it in a DataFrame to easily grab our top 150 companies
    df_companies = pd.DataFrame(companies)
    
    # Let's take the first 150 companies
    target_companies = df_companies.head(150)
    print(f"Successfully loaded {len(target_companies)} companies.")
    print(target_companies.head())
else:
    print(f"Failed to fetch tickers. Status code: {response.status_code}")

Successfully loaded 150 companies.
          cik ticker           title
0  0001045810   NVDA     NVIDIA CORP
1  0000320193   AAPL      Apple Inc.
2  0001652044  GOOGL   Alphabet Inc.
3  0000789019   MSFT  MICROSOFT CORP
4  0001018724   AMZN  AMAZON COM INC


Web Scraping 8-K Filings
Fetch and parse the 8-K filiings to go with the extracted CIKs using BeautifulSoup. Uses the SEC EDGAR Atom feed URL to locate the latest 8-K filing for a given company and extract its raw text. The code sets a timeout limit on the HTTP request and wraps the logic in a retry loop so a single dropped connection does not crash the entire 150-document batch.

In [6]:
# Fetch and parse the 8-K filings.

import warnings
from bs4 import XMLParsedAsHTMLWarning

# Hide the XML parser warning for clean output
warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

def get_latest_8k_text(cik, headers, max_retries=5):
    search_url = f"https://www.sec.gov/cgi-bin/browse-edgar?action=getcompany&CIK={cik}&type=8-K&count=1&output=atom"
    
    for attempt in range(max_retries):
        try:
            response = requests.get(search_url, headers=headers, timeout=15)
            if response.status_code != 200:
                return None
                
            soup = BeautifulSoup(response.content, 'html.parser')
            entry = soup.find('entry')
            
            if not entry:
                return None
                
            index_link = entry.find('link')['href']
            full_text_link = index_link.replace('-index.htm', '.txt')
            
            doc_response = requests.get(full_text_link, headers=headers, timeout=15)
            doc_soup = BeautifulSoup(doc_response.content, 'html.parser')
            
            return doc_soup.get_text(separator=' ', strip=True)
            
        except requests.exceptions.RequestException as e:
            print(f" Timeout on attempt {attempt+1}. Retrying in 5 seconds...")
            time.sleep(5)
            
    return None

# Test the updated function
sample_cik = target_companies.iloc[0]['cik']
sample_company = target_companies.iloc[0]['title']

print(f"Fetching 8-K for {sample_company} (CIK: {sample_cik})...")
sample_text = get_latest_8k_text(sample_cik, headers)

if sample_text:
    print(f"Success! Extracted {len(sample_text)} characters.")
    print(sample_text[:500] + "...")
else:
    print("No 8-K found or extraction failed.")

Fetching 8-K for NVIDIA CORP (CIK: 0001045810)...
Success! Extracted 54759 characters.
0001045810-26-000078.txt : 20260903 0001045810-26-000078.hdr.sgml : 20260903 20260903080356
ACCESSION NUMBER:		0001045810-26-000078
CONFORMED SUBMISSION TYPE:	8-K
PUBLIC DOCUMENT COUNT:		11
CONFORMED PERIOD OF REPORT:	20260902
ITEM INFORMATION:		Other Events
FILED AS OF DATE:		20260903
DATE AS OF CHANGE:		20260903

FILER:

	COMPANY DATA:	
		COMPANY CONFORMED NAME:			NVIDIA CORP
		CENTRAL INDEX KEY:			0001045810
		STANDARD INDUSTRIAL CLASSIFICATION:	SEMICONDUCTORS & RELATED DEVICES [3674]
		ORGAN...


Connect Python to DeepSeek-R1 (local LLM)
Function to send the scraped text to local Ollama instance and enforce the rules (Named Entity Recognition, product description < 180 chars, and the | delimiter) by sending HTTP POST requests directly to its default API port.

In [7]:
def extract_product_info(text, model="deepseek-r1:8b"):
    # Truncate text to fit within standard local LLM context windows
    # The SEC header and introductory paragraphs typically contain the announcement
    prompt_text = text[:8000] 
    
    prompt = f"""
    Analyze the following SEC 8-K filing excerpt and extract the new product announcement using Named Entity Recognition.
    Extract exactly these fields: Company Name, Stock Ticker, Filing Time, New Product, and Product Description.
    Constraint 1: The Product Description must be strictly under 180 characters.
    Constraint 2: Output ONLY a single line with the values separated by pipes (|), matching this format exactly:
    Company Name | Stock Name | Filing Time | New Product | Product Description
    
    Filing Text:
    {prompt_text}
    """
    
    url = "http://localhost:11434/api/generate"
    payload = {
        "model": model,
        "prompt": prompt,
        "stream": False
    }
    
    try:
        response = requests.post(url, json=payload)
        if response.status_code == 200:
            return response.json().get('response', '').strip()
        return f"API Error: {response.status_code}"
    except Exception as e:
        return f"Connection Failed. Ensure Ollama is running. Error: {e}"

# Test the pipeline on your NVIDIA sample
print("Analyzing NVIDIA 8-K with deepseek-r1:8b...")
llm_result = extract_product_info(sample_text)
print(llm_result)

Analyzing NVIDIA 8-K with deepseek-r1:8b...
NVIDIA CORP | NVDA | September 3, 2026 | Equity-based retention program | Equity-based retention program for Hugging Face employees joining NVIDIA, with up to approximately $1.0 billion in equity.


The Processing Loop & CSV Export
Loop through the 150 documents, collect the structured data and export it to a CSV.

In [8]:
# This list will hold all of our successfully extracted dictionaries
extracted_data = []

print(f"Starting analysis of {len(target_companies)} companies. This may take a while...")

# Loop through each company in our target list
for index, row in target_companies.iterrows():
    cik = row['cik']
    ticker = row['ticker']
    company_name = row['title']
    
    # Fetch 8-K Text
    text = get_latest_8k_text(cik, headers)
    
    # Pause for 1 seconds to respect SEC rate limits (max 10 requests/sec)
    time.sleep(1)
    
    if not text:
        print(f"[{index+1}/{len(target_companies)}] Skipped {ticker} - No 8-K found.")
        continue
        
    # Extract with LLM
    llm_response = extract_product_info(text)
    
    # Clean up the response (Remove <think> tags if DeepSeek-R1 generated them)
    clean_response = re.sub(r'<think>.*?</think>', '', llm_response, flags=re.DOTALL).strip()
    
    # Parse the pipe-delimited string
    parts = [p.strip() for p in clean_response.split('|')]
    
    # Ensure the model gave us exactly 5 fields before saving
    if len(parts) == 5:
        extracted_data.append({
            'company_name': parts[0],
            'stock_name': parts[1],
            'filing time': parts[2],
            'new_product': parts[3],
            'product_description': parts[4]
        })
        print(f"[{index+1}/{len(target_companies)}] Processed {ticker}: {parts[3]}")
    else:
        print(f"[{index+1}/{len(target_companies)}] Failed to parse {ticker}. Raw output: {clean_response}")

# Export to CSV matching the assignment requirements
df_results = pd.DataFrame(extracted_data)
csv_filename = "extracted_8k_products.csv"
df_results.to_csv(csv_filename, index=False)

print(f"\nExtraction complete! Successfully processed {len(df_results)} documents.")
print(f"Data saved to {csv_filename}")

Starting analysis of 150 companies. This may take a while...
[1/150] Processed NVDA: Hugging Face platform and its open-source ecosystem
[2/150] Processed AAPL: Common Stock
[3/150] Processed GOOGL: Senior Notes
[4/150] Processed MSFT: Agents and Infra, Devices and Consumer
[5/150] Processed AMZN: 5.200% notes due 2029
[6/150] Processed META: Founder Alignment Agreement
[7/150] Processed AVGO: AI racks based on custom AI accelerators
[8/150] Processed TSLA: Cybercab
[9/150] Processed BRK-B: 
[10/150] Processed LLY: retatrutide
[11/150] Processed MU: AI-driven demand
[12/150] Skipped SKHY - No 8-K found.
[13/150] Processed WMT: U.S. Notes and Canadian Notes
[14/150] Processed JPM: Floating Rate Notes due 2030, 2030 Fixed-to-Floating Rate Notes, 2037 Fixed-to-Floating Rate Notes
[15/150] Processed AMD: (No new product announced in this filing)
[16/150] Processed V: 
[17/150] Processed XOM: 0.524% Notes due 2028 (XOM28), 0.835% Notes due 2032 (XOM32), 1.408% Notes due 2039 (XOM39A)
[18/15